# Anomaly Segmentation (Step 7)

In [ ]:
!git clone https://github.com/letiBri/MaskArchitectureAnomaly_CourseProject.git
%cd MaskArchitectureAnomaly_CourseProject/eval

fatal: destination path 'MaskArchitectureAnomaly_CourseProject' already exists and is not an empty directory.
/content/MaskArchitectureAnomaly_CourseProject/eval


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import sys # Allows importing from the 'eomt' folder
sys.path.append('/content/MaskArchitectureAnomaly_CourseProject')
sys.path.append('/content/MaskArchitectureAnomaly_CourseProject/eval')

In [ ]:
!pip install ood-metrics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 98.0 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jax 0.

In [ ]:
import os
import cv2
import glob
import torch
import random
from PIL import Image
import numpy as np
from erfnet import ERFNet
import os.path as osp
from argparse import ArgumentParser
from ood_metrics import fpr_at_95_tpr, calc_metrics, plot_roc, plot_pr,plot_barcode
from sklearn.metrics import roc_auc_score, roc_curve, auc, precision_recall_curve, average_precision_score
from torchvision.transforms import Compose, Resize, ToTensor, Normalize

In [ ]:
seed = 42

# general reproducibility
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

NUM_CHANNELS = 3
NUM_CLASSES = 20
# gpu training specific
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = True

input_transform = Compose(
    [
        Resize((512, 1024), Image.BILINEAR),
        ToTensor(),
        # Normalize([.485, .456, .406], [.229, .224, .225]),
    ]
)

target_transform = Compose(
    [
        Resize((512, 1024), Image.NEAREST),
    ]
)


In [ ]:
parser = ArgumentParser()
parser.add_argument(
    "--input",
    default="/content/drive/MyDrive/CourseProjectAnomaly/Anomaly_Validation_Datasets/Validation_Dataset/FS_LostFound_full/images/*.png",
    nargs="+",
    help="A list of space separated input images; "
    "or a single glob pattern such as 'directory/*.jpg'",
)
parser.add_argument('--loadDir',default="../trained_models/")
parser.add_argument('--loadWeights', default="erfnet_pretrained.pth")
parser.add_argument('--loadModel', default="erfnet.py")
parser.add_argument('--subset', default="val")  #can be val or train (must have labels)
#parser.add_argument('--datadir', default="???????")
parser.add_argument('--num-workers', type=int, default=4)
parser.add_argument('--batch-size', type=int, default=1)
parser.add_argument('--cpu', action='store_true')
args = parser.parse_args(args=[])
anomaly_score_list_maxlogit = []
anomaly_score_list_msp = []
anomaly_score_list_maxentropy = []
ood_gts_list = []

if not os.path.exists('results.txt'):
    open('results.txt', 'w').close()
file = open('results.txt', 'a')

modelpath = args.loadDir + args.loadModel
weightspath = args.loadDir + args.loadWeights

print ("Loading model: " + modelpath)
print ("Loading weights: " + weightspath)

model = ERFNet(NUM_CLASSES)

if (not args.cpu):
    model = torch.nn.DataParallel(model).cuda()

def load_my_state_dict(model, state_dict):  #custom function to load model when not all dict elements
    own_state = model.state_dict()
    for name, param in state_dict.items():
        if name not in own_state:
            if name.startswith("module."):
                own_state[name.split("module.")[-1]].copy_(param)
            else:
                print(name, " not loaded")
                continue
        else:
            own_state[name].copy_(param)
    return model

model = load_my_state_dict(model, torch.load(weightspath, map_location=lambda storage, loc: storage))
print ("Model and weights LOADED successfully")
model.eval()

Loading model: ../trained_models/erfnet.py
Loading weights: ../trained_models/erfnet_pretrained.pth
Model and weights LOADED successfully


DataParallel(
  (module): ERFNet(
    (encoder): Encoder(
      (initial_block): DownsamplerBlock(
        (conv): Conv2d(3, 13, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
        (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
        (bn): BatchNorm2d(16, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
      )
      (layers): ModuleList(
        (0): DownsamplerBlock(
          (conv): Conv2d(16, 48, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
          (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
          (bn): BatchNorm2d(64, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
        )
        (1-5): 5 x non_bottleneck_1d(
          (conv3x1_1): Conv2d(64, 64, kernel_size=(3, 1), stride=(1, 1), padding=(1, 0))
          (conv1x3_1): Conv2d(64, 64, kernel_size=(1, 3), stride=(1, 1), padding=(0, 1))
          (bn1): BatchNorm2d(64, eps=0.001, momentum=0.1, affine=True

In [ ]:
#for path in glob.glob(os.path.expanduser(str(args.input[0]))):

# Controlliamo se args.input è una lista o una stringa
input_pattern = args.input[0] if isinstance(args.input, list) else args.input

for path in glob.glob(os.path.expanduser(str(input_pattern))):
    print(path)
    images = input_transform((Image.open(path).convert('RGB'))).unsqueeze(0).float().cuda()
    #images = images.permute(0,3,1,2)
    with torch.no_grad():
        result = model(images)

    #compute max logit
    anomaly_result_maxlogit = 1.0 - np.max(result.squeeze(0).data.cpu().numpy(), axis=0)

    #compute MSP
    probs = torch.nn.functional.softmax(result, dim=1)
    max_probs, _ = torch.max(probs.squeeze(0), dim=0)
    anomaly_result_msp = 1.0 - max_probs.data.cpu().numpy()

    #compute Max Entropy
    probs = torch.nn.functional.softmax(result, dim=1).squeeze(0)
    epsilon = 1e-10
    entropy = -torch.sum(probs * torch.log(probs + epsilon), dim=0)
    anomaly_result_maxentropy = entropy.data.cpu().numpy()

    pathGT = path.replace("images", "labels_masks")
    if "RoadObsticle21" in pathGT:
        pathGT = pathGT.replace("webp", "png")
    if "fs_static" in pathGT:
        pathGT = pathGT.replace("jpg", "png")
    if "RoadAnomaly" in pathGT:
        pathGT = pathGT.replace("jpg", "png")
    if "FS_LostFound_full" in pathGT:
        pathGT = pathGT.replace("jpg", "png")

    mask = Image.open(pathGT)
    mask = target_transform(mask)
    ood_gts = np.array(mask)

    if "RoadAnomaly" in pathGT:
      #in RoadAnomaly 2 indica anomalia --> viene trasformato in 1 per uniformare
        ood_gts = np.where((ood_gts==2), 1, ood_gts)
    #if "LostFound" in pathGT:
    #    ood_gts = np.where((ood_gts==0), 255, ood_gts)
    #    ood_gts = np.where((ood_gts==1), 0, ood_gts)
    #    ood_gts = np.where((ood_gts>1)&(ood_gts<201), 1, ood_gts)

    #if "Streethazard" in pathGT:
     #   ood_gts = np.where((ood_gts==14), 255, ood_gts)
      #  ood_gts = np.where((ood_gts<20), 0, ood_gts)
       # ood_gts = np.where((ood_gts==255), 1, ood_gts)

    if 1 not in np.unique(ood_gts):
        #print("no anomalie")
        continue   #se non c'è nessuna anomalia, allora salta l'immagine e passa a valutare la successiva
    else:
          print('anomalia')
          ood_gts_list.append(ood_gts)
          anomaly_score_list_maxlogit.append(anomaly_result_maxlogit)
          anomaly_score_list_msp.append(anomaly_result_msp)
          anomaly_score_list_maxentropy.append(anomaly_result_maxentropy)

    del result, anomaly_result_maxlogit,anomaly_result_msp, anomaly_result_maxentropy, ood_gts, mask
    torch.cuda.empty_cache()

file.write( "------\n")

/content/drive/MyDrive/CourseProjectAnomaly/Anomaly_Validation_Datasets/Validation_Dataset/FS_LostFound_full/images/48.png
anomalia
/content/drive/MyDrive/CourseProjectAnomaly/Anomaly_Validation_Datasets/Validation_Dataset/FS_LostFound_full/images/63.png
anomalia
/content/drive/MyDrive/CourseProjectAnomaly/Anomaly_Validation_Datasets/Validation_Dataset/FS_LostFound_full/images/88.png
anomalia
/content/drive/MyDrive/CourseProjectAnomaly/Anomaly_Validation_Datasets/Validation_Dataset/FS_LostFound_full/images/89.png
anomalia
/content/drive/MyDrive/CourseProjectAnomaly/Anomaly_Validation_Datasets/Validation_Dataset/FS_LostFound_full/images/77.png
anomalia
/content/drive/MyDrive/CourseProjectAnomaly/Anomaly_Validation_Datasets/Validation_Dataset/FS_LostFound_full/images/71.png
anomalia
/content/drive/MyDrive/CourseProjectAnomaly/Anomaly_Validation_Datasets/Validation_Dataset/FS_LostFound_full/images/59.png
anomalia
/content/drive/MyDrive/CourseProjectAnomaly/Anomaly_Validation_Datasets/Vali

7

In [ ]:
ood_gts = np.array(ood_gts_list)
anomaly_scores_maxlogit = np.array(anomaly_score_list_maxlogit)
anomaly_scores_msp = np.array(anomaly_score_list_msp)
anomaly_scores_maxentropy = np.array(anomaly_score_list_maxentropy)

ood_mask = (ood_gts == 1)
ind_mask = (ood_gts == 0)

# max logit
ood_out = anomaly_scores_maxlogit[ood_mask]
ind_out = anomaly_scores_maxlogit[ind_mask]

ood_label = np.ones(len(ood_out))
ind_label = np.zeros(len(ind_out))

val_out = np.concatenate((ind_out, ood_out))
val_label = np.concatenate((ind_label, ood_label))

prc_auc = average_precision_score(val_label, val_out)
fpr = fpr_at_95_tpr(val_out, val_label)

print(f'AUPRC score maxlogit: {prc_auc*100.0}')
print(f'FPR@TPR95 maxlogit: {fpr*100.0}')

file.write(('    AUPRC score maxlogit:' + str(prc_auc*100.0) + '   FPR@TPR95 maxlogit:' + str(fpr*100.0) ))

#MSP
ood_out = anomaly_scores_msp[ood_mask]
ind_out = anomaly_scores_msp[ind_mask]

ood_label = np.ones(len(ood_out))
ind_label = np.zeros(len(ind_out))

val_out = np.concatenate((ind_out, ood_out))
val_label = np.concatenate((ind_label, ood_label))

prc_auc = average_precision_score(val_label, val_out)
fpr = fpr_at_95_tpr(val_out, val_label)

print(f'AUPRC score msp: {prc_auc*100.0}')
print(f'FPR@TPR95 msp: {fpr*100.0}')

file.write(('    AUPRC score msp:' + str(prc_auc*100.0) + '   FPR@TPR95 msp:' + str(fpr*100.0) ))

#max entropy
ood_out = anomaly_scores_maxentropy[ood_mask]
ind_out = anomaly_scores_maxentropy[ind_mask]

ood_label = np.ones(len(ood_out))
ind_label = np.zeros(len(ind_out))

val_out = np.concatenate((ind_out, ood_out))
val_label = np.concatenate((ind_label, ood_label))

prc_auc = average_precision_score(val_label, val_out)
fpr = fpr_at_95_tpr(val_out, val_label)

print(f'AUPRC score maxentropy: {prc_auc*100.0}')
print(f'FPR@TPR95 maxentropy: {fpr*100.0}')

file.write(('    AUPRC score maxentropy:' + str(prc_auc*100.0) + '   FPR@TPR95 maxentropy:' + str(fpr*100.0) ))

file.close()

AUPRC score maxlogit: 3.3014401838550618
FPR@TPR95 maxlogit: 45.494847365050845
AUPRC score msp: 1.749039030843141
FPR@TPR95 msp: 50.594030268678914
AUPRC score maxentropy: 2.5839564298873796
FPR@TPR95 maxentropy: 50.162776109239616


# Evaluation mean Intersection Over Union

In [ ]:
# Crea la cartella di destinazione
!mkdir -p /content/cityscapes_data

# Unzip delle immagini (usa il nome esatto del tuo file zip)
!unzip -q /content/drive/MyDrive/CourseProjectAnomaly/cityscapes/leftImg8bit_trainvaltest1.zip -d /content/cityscapes_data/

# Unzip delle maschere
!unzip -q /content/drive/MyDrive/CourseProjectAnomaly/cityscapes/gtFine_trainvaltest1.zip -d /content/cityscapes_data/

replace /content/cityscapes_data/README? [y]es, [n]o, [A]ll, [N]one, [r]ename: A


In [ ]:
# 1. Installiamo gli script ufficiali di Cityscapes
!pip install cityscapesscripts

# 2. Diciamo agli script dove si trova la nostra cartella scompattata
import os
os.environ['CITYSCAPES_DATASET'] = '/content/cityscapes_data/'

# 3. Lanciamo il convertitore ufficiale
!csCreateTrainIdLabelImgs

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.6/78.6 kB 4.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 473.6/473.6 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 8.5 MB/s eta 0:00:00
  Created wheel for typing: filename=typing-3.7.4.3-py3-none-any.whl size=26304 sha256=c81f79f583ade8079ef9e942f88746c112b04d15260237080841e3def5b0cf85
  Stored in directory: /root/.cache/pip/wheels/12/98/52/2bffe242a9a487f00886e43b8ed8dac46456702e11a0d6abef
Successfully built typing


Processing 5000 annotation files
Progress: 100.0 % 

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
import os
import importlib
import time

from PIL import Image
from argparse import ArgumentParser

from torch.autograd import Variable
from torch.utils.data import DataLoader
from torchvision.transforms import Compose, CenterCrop, Normalize, Resize
from torchvision.transforms import ToTensor, ToPILImage

from dataset import cityscapes
from erfnet import ERFNet
from transform import Relabel, ToLabel, Colorize
from iouEval import iouEval, getColorEntry

image_transform = ToPILImage()
input_transform_cityscapes = Compose([
    Resize(512, Image.BILINEAR),
    ToTensor(),
])
target_transform_cityscapes = Compose([
    Resize(512, Image.NEAREST),
    ToLabel(),
    Relabel(255, 19),   #ignore label to 19
])


parser = ArgumentParser()

parser.add_argument('--state')

parser.add_argument('--loadDir',default="../trained_models/")
parser.add_argument('--loadWeights', default="erfnet_pretrained.pth")
parser.add_argument('--loadModel', default="erfnet.py")
parser.add_argument('--subset', default="val")  #can be val or train (must have labels)
parser.add_argument('--datadir', default="/content/cityscapes_data/")
parser.add_argument('--num-workers', type=int, default=2) # diminuito da 4 a 2
parser.add_argument('--batch-size', type=int, default=1)
parser.add_argument('--cpu', action='store_true')
args = parser.parse_args(args=[])


if(not os.path.exists(args.datadir)):
    print ("Error: datadir could not be loaded")


loader = DataLoader(cityscapes(args.datadir, input_transform_cityscapes, target_transform_cityscapes, subset=args.subset), num_workers=args.num_workers, batch_size=args.batch_size, shuffle=False)


iouEvalVal = iouEval(NUM_CLASSES)

start = time.time()


#-----
# model = ERFNet(NUM_CLASSES)
# if not args.cpu:
#     model = torch.nn.DataParallel(model).cuda()

# # Caricamento dei pesi (fondamentale!)
# checkpoint = torch.load(weightspath, map_location=lambda storage, loc: storage)
# model.load_state_dict(checkpoint)
model.eval()
# print("Modello caricato correttamente.")
#-----


for step, (images, labels, filename, filenameGt) in enumerate(loader):
    if (not args.cpu):
        images = images.cuda()
        labels = labels.cuda()

    inputs = Variable(images)
    with torch.no_grad():
        outputs = model(inputs)

    iouEvalVal.addBatch(outputs.max(1)[1].unsqueeze(1).data, labels)

    filenameSave = filename[0].split("leftImg8bit/")[1]

    print (step, filenameSave)


iouVal, iou_classes = iouEvalVal.getIoU()

iou_classes_str = []
for i in range(iou_classes.size(0)):
    iouStr = getColorEntry(iou_classes[i])+'{:0.2f}'.format(iou_classes[i]*100) + '\033[0m'
    iou_classes_str.append(iouStr)

print("---------------------------------------")
print("Took ", time.time()-start, "seconds")
print("=======================================")
#print("TOTAL IOU: ", iou * 100, "%")
print("Per-Class IoU:")
print(iou_classes_str[0], "Road")
print(iou_classes_str[1], "sidewalk")
print(iou_classes_str[2], "building")
print(iou_classes_str[3], "wall")
print(iou_classes_str[4], "fence")
print(iou_classes_str[5], "pole")
print(iou_classes_str[6], "traffic light")
print(iou_classes_str[7], "traffic sign")
print(iou_classes_str[8], "vegetation")
print(iou_classes_str[9], "terrain")
print(iou_classes_str[10], "sky")
print(iou_classes_str[11], "person")
print(iou_classes_str[12], "rider")
print(iou_classes_str[13], "car")
print(iou_classes_str[14], "truck")
print(iou_classes_str[15], "bus")
print(iou_classes_str[16], "train")
print(iou_classes_str[17], "motorcycle")
print(iou_classes_str[18], "bicycle")
print("=======================================")
iouStr = getColorEntry(iouVal)+'{:0.2f}'.format(iouVal*100) + '\033[0m'
print ("MEAN IoU: ", iouStr, "%")


/content/cityscapes_data/leftImg8bit/val /content/cityscapes_data/gtFine/val
0 val/frankfurt/frankfurt_000000_000294_leftImg8bit.png
1 val/frankfurt/frankfurt_000000_000576_leftImg8bit.png
2 val/frankfurt/frankfurt_000000_001016_leftImg8bit.png
3 val/frankfurt/frankfurt_000000_001236_leftImg8bit.png
4 val/frankfurt/frankfurt_000000_001751_leftImg8bit.png
5 val/frankfurt/frankfurt_000000_002196_leftImg8bit.png
6 val/frankfurt/frankfurt_000000_002963_leftImg8bit.png
7 val/frankfurt/frankfurt_000000_003025_leftImg8bit.png
8 val/frankfurt/frankfurt_000000_003357_leftImg8bit.png
9 val/frankfurt/frankfurt_000000_003920_leftImg8bit.png
10 val/frankfurt/frankfurt_000000_004617_leftImg8bit.png
11 val/frankfurt/frankfurt_000000_005543_leftImg8bit.png
12 val/frankfurt/frankfurt_000000_005898_leftImg8bit.png
13 val/frankfurt/frankfurt_000000_006589_leftImg8bit.png
14 val/frankfurt/frankfurt_000000_007365_leftImg8bit.png
15 val/frankfurt/frankfurt_000000_008206_leftImg8bit.png
16 val/frankfurt/fran